In [1]:
%run ../resources/thedatasociety.py
%run ../resources/device.py

In [2]:
from IPython.display import display
import ipywidgets as widgets
import os

# Componente para simulação de um sensor

<!--
```bash
IoT_sensor(<name/id>, <grandeza física >, <unidade de medida>, <menor valor>, <maior valor possível>, <intervalo entre leituras (segundos)>)
```

Exemplo de sensor de pressão:

```python
sensor_pressao = IoT_sensor("32", "pressao", "bar", 20, 35, 5)

```
-->

Componentes `IoT_sensor` podem se conectar a componentes do tipo `IoT_mqtt_publisher` para publicar, em um tópico, mensagens referentes às leituras feitas pelo sensor. Por exemplo, o sensor do exemplo acima produziu a seguinte mensagem no tópico `sensor/32/pressao`:

```python
 {
 	"source": "sensor",
 	"name": "32",
 	"type": "mode-r2d2",
 	"body": {
 		"timestamp": "2019-08-17 17:02:15",
 		"dimension": "pressao",
 		"value": 25.533895448246717,
 		"unity": "bar"
 	}
 }
```

Simuladores podem publicar mensagens aleatórias de tempos em tempos.


# Instanciando Simuladores de Sensores

In [3]:
sensor_1 = IoT_sensor(name="1", dimension="Temperatura",     unity="°C",  min_value=25, max_value=35,   pooling_interval=2)
sensor_2 = IoT_sensor(name="2", dimension="Umidade",         unity="%",   min_value=40, max_value=65,   pooling_interval=3)
sensor_3 = IoT_sensor(name="3", dimension="BateriaRestante", unity="%",   min_value=5,  max_value=100,  pooling_interval=4)
sensor_4 = IoT_sensor(name="4", dimension="SinalWifi",       unity="dBm", min_value=-30, max_value=-90, pooling_interval=2)

# Instanciando Componente de Publicação de Mensagens no MQTT

In [4]:
publisher = IoT_mqtt_publisher("127.0.0.1", 1883)

Connected.


### Conectando os Componentes 

In [5]:
sensor_1.connect(publisher)
sensor_2.connect(publisher)
sensor_3.connect(publisher)
sensor_4.connect(publisher)

# Escutando o barramento

Abra um terminal e digite:

```bash

mosquitto_sub -t "sensor/#" -v

```

# Construindo um "dashboard" local

### Criando o consumidor_1 e seus widgets

In [6]:
widget_1 = widgets.FloatProgress(min=0, max=40.0, bar_style='', orientation='vertical'); widget_1_label = widgets.Label()

consumer_1 = IoTSensorConsumer("127.0.0.1",1883,"sensor/1/+")

### Criando o consumidor_2 e seus widgets

In [7]:
widget_2 = widgets.FloatProgress(min=0, max=90.0, bar_style='warning', orientation='vertical'); widget_2_label = widgets.Label()

consumer_2 = IoTSensorConsumer("127.0.0.1",1883,"sensor/2/+")

### Criando o consumidor_3 e seus widgets

In [8]:
widget_3  = widgets.FloatProgress(min=0, max=100.0, bar_style='info', orientation='vertical'); widget_3_label = widgets.Label()

consumer_3 = IoTSensorConsumer("127.0.0.1",1883,"sensor/3/+")

### Criando o consumidor_4 e seus widgets

In [9]:
widget_4  = widgets.FloatProgress(min=-95.0, max=-30, value=-95, bar_style='success', orientation='vertical'); widget_4_label = widgets.Label()

consumer_4 = IoTSensorConsumer("127.0.0.1",1883,"sensor/4/+")

## Organizando os componentes visualmente

In [10]:
separator = widgets.Label(value="_____")
col_1 = widgets.VBox([widget_1, widget_1_label])
col_2 = widgets.VBox([widget_2, widget_2_label])
col_3 = widgets.VBox([widget_3, widget_3_label])
col_4 = widgets.VBox([widget_4, widget_4_label])
row_1 = widgets.HBox([separator, col_1, separator, col_2, separator, col_3, separator, col_4])
display(row_1)

### Conectando componentes visuais e seus respectivos consumidores

In [11]:
consumer_1.start_consuming(widget_1, widget_1_label)
consumer_2.start_consuming(widget_2, widget_2_label)
consumer_3.start_consuming(widget_3, widget_3_label)
consumer_4.start_consuming(widget_4, widget_4_label)

Inscrito com sucesso em sensor/1/+
Inscrito com sucesso em sensor/2/+
Inscrito com sucesso em sensor/3/+
Inscrito com sucesso em sensor/4/+


# Consultando InfluxDB

In [12]:
! influxdb3 create database meu_db

Database "meu_db" created successfully


In [13]:
! influxdb3 show databases

+---------------+
| iox::database |
+---------------+
| _internal     |
| meu_db        |
+---------------+


In [14]:
! influxdb3 query --database exemplo_iot "show tables"

Query command failed: server responded with error [404 Not Found]: {"error":"query error: database not found: exemplo_iot"}


In [15]:
! influxdb3 query --database exemplo_iot --language influxql "SHOW MEASUREMENTS"

Query command failed: server responded with error [404 Not Found]: {"error":"query error: database not found: exemplo_iot"}


In [16]:
! influxdb3 show system --database exemplo_iot summary

Show command failed: client error: server responded with error [404 Not Found]: {"error":"query error: database not found: exemplo_iot"}


In [17]:
! influxdb3 query --database exemplo_iot "SELECT * FROM cliente_a ORDER BY time DESC LIMIT 10 "

Query command failed: server responded with error [404 Not Found]: {"error":"query error: database not found: exemplo_iot"}


In [18]:
! influxdb3 query --database exemplo_iot "SELECT * FROM cliente_a WHERE sensor_id = '1' ORDER BY time DESC LIMIT 10 "

Query command failed: server responded with error [404 Not Found]: {"error":"query error: database not found: exemplo_iot"}


# Acessando o Grafana

A célula abaixo permite acessar a instância do Grafana que está rodando nesta sessão. Verifique a sua URL de acordo com o servidor utilizado.

In [20]:
binderhub_urls= ['binder.curvenote.dev', 'hub.ovh2.mybinder.org', 'notebooks.gesis.org', 'mybinder.org', 'localhost:8888']

list_urls(binderhub_urls)


O Grafana está disponível em alguma das URLs abaixo, dependendo do servidor escolhido: 



## Adicione um data source

Go to the setting menu (left cog on the vertical bar) and click to add a data source.

![](../resources/images/screenshot-grafana0.png)

Select InfluxDB and then set the following parameters:


- Name: influxdb
- Type: influxdb
- URL: http://localhost:8086/
- Database: telegraf
- User: telegraf
- Password: 'telegraf'

Then click `save & test`.

![](../resources/images/screenshot-grafana1.png)

## Importe um dashboard ou crie o seu

Grafana provides the repository for grafana plugins and dashboards.

- [Grafana Plugins](https://grafana.com/plugins)
- [Grafana Dashboards](https://grafana.com/dashboards)


For instance, go to [https://grafana.com/dashboards/5955](https://grafana.com/dashboards/5955) and click the 'Copy the ID to Clipboard' button.

Click on the `+` option on the left bar and select `import`. Paste the ID, select the Data Source and you will be done!